# D5 Climate Risk Workflow — GeoPrompt

Climate zone risk scoring, raster algebra, and adaptation scenario analysis.

In [ ]:
from __future__ import annotations
import json, os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen
import matplotlib.pyplot as plt

OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "1") == "1"

def fetch_json(url, fallback):
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})
        with urlopen(req, timeout=6) as r:
            return json.loads(r.read().decode("utf-8"))
    except (URLError, TimeoutError, ValueError):
        return fallback

def fetch_first_json(urls, validator, fallback):
    for url in urls:
        payload = fetch_json(url, None)
        if payload is not None and validator(payload):
            return payload, url, True
    return fallback, "fallback", False

import geoprompt as gp
from geoprompt import GeoPromptFrame, write_geojson
from geoprompt.raster import raster_algebra, raster_slope_aspect
from geoprompt.tools import build_scenario_report, export_scenario_report
print("Imports OK")


## Section A: Pull Data Sources

In [ ]:
climate = {"features": [{"id": "fallback-climate"}]}
weather = {"properties": {"forecast": "fallback"}}
forecast = {"hourly": {"temperature_2m": [0.0]}}

climate, cl_src, cl_live = fetch_first_json(
    [
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson",
        "https://api.github.com/repos/ecmwf/cdsapi",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("features") or d.get("id")),
    climate,
)
weather, wx_src, wx_live = fetch_first_json(
    [
        "https://api.weather.gov/points/39.00,-98.00",
        "https://api.weather.gov/points/38.90,-77.03",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("properties", {}).get("forecast")),
    weather,
)
forecast, fc_src, fc_live = fetch_first_json(
    [
        "https://api.open-meteo.com/v1/forecast?latitude=39.00&longitude=-98.00&hourly=temperature_2m&forecast_days=1",
        "https://api.open-meteo.com/v1/forecast?latitude=38.90&longitude=-77.03&hourly=temperature_2m&forecast_days=1",
    ],
    lambda d: isinstance(d, dict) and len(d.get("hourly", {}).get("temperature_2m", [])) > 0,
    forecast,
)

climate_count = len(climate.get("features", [])) if isinstance(climate, dict) else 0
if climate_count == 0 and isinstance(climate, dict) and climate.get("id"):
    climate_count = 1
print(f"Climate records: {climate_count} | live={cl_live} | source={cl_src}")
print(f"NOAA forecast exists: {bool(weather.get('properties', {}).get('forecast'))} | live={wx_live} | source={wx_src}")
print(f"Open-Meteo hourly points: {len(forecast.get('hourly', {}).get('temperature_2m', []))} | live={fc_live} | source={fc_src}")


## Section B: Spatial Analysis

In [ ]:
RAW_ZONES = [
    {"zone_id": "Z1", "heat_index": 0.78, "drought_idx": 0.65, "sea_level_risk": 0.30,
     "pop_density": 1200.0, "geometry": {"type": "Point", "coordinates": [-98.10, 39.20]}},
    {"zone_id": "Z2", "heat_index": 0.62, "drought_idx": 0.72, "sea_level_risk": 0.15,
     "pop_density":  850.0, "geometry": {"type": "Point", "coordinates": [-97.80, 38.90]}},
    {"zone_id": "Z3", "heat_index": 0.85, "drought_idx": 0.55, "sea_level_risk": 0.80,
     "pop_density": 3400.0, "geometry": {"type": "Point", "coordinates": [-97.50, 39.40]}},
    {"zone_id": "Z4", "heat_index": 0.71, "drought_idx": 0.80, "sea_level_risk": 0.20,
     "pop_density":  550.0, "geometry": {"type": "Point", "coordinates": [-98.40, 38.70]}},
    {"zone_id": "Z5", "heat_index": 0.90, "drought_idx": 0.45, "sea_level_risk": 0.60,
     "pop_density": 2100.0, "geometry": {"type": "Point", "coordinates": [-97.65, 39.60]}},
]
ADAPTATION_SITES = [
    {"site_id": "AS1", "capacity_mw": 120, "geometry": {"type": "Point", "coordinates": [-97.90, 39.10]}},
    {"site_id": "AS2", "capacity_mw":  90, "geometry": {"type": "Point", "coordinates": [-98.20, 39.30]}},
]

enriched = []
for row in RAW_ZONES:
    risk = round(float(row["heat_index"]) * 0.35 + float(row["drought_idx"]) * 0.35
                 + float(row["sea_level_risk"]) * 0.30, 4)
    risk_tier = "HIGH" if risk > 0.65 else ("MED" if risk > 0.45 else "LOW")
    enriched.append({**row, "composite_risk": risk, "risk_tier": risk_tier})

zones_frame = GeoPromptFrame(enriched, geometry_column="geometry", crs="EPSG:4326")
sites_frame = GeoPromptFrame(ADAPTATION_SITES, geometry_column="geometry", crs="EPSG:4326")

# 1. Nearest neighbors between climate zones
neighbors = zones_frame.nearest_neighbors(id_column="zone_id", k=1, distance_method="haversine")
print("Nearest zone pairs:")
for nb in neighbors:
    print(f"  {nb['origin']} -> {nb['neighbor']}  dist={nb['distance']:.4f}")

# 2. Query radius: zones near Z3 (highest risk)
near_z3 = zones_frame.query_radius("Z3", max_distance=0.6, id_column="zone_id")
print(f"\nZones within 0.6 deg of Z3: {[r['zone_id'] for r in near_z3.to_records()]}")

# 3. Buffer: climate impact zones
buffers = zones_frame.buffer(0.2)
print(f"Climate buffer zones: {len(buffers)} polygons")

# 4. Proximity join: zones to adaptation sites within 0.7 deg
pj = zones_frame.proximity_join(sites_frame, max_distance=0.7, how="left")
print(f"\nZone->site proximity assignments: {len(pj)} rows")
for row in pj.to_records()[:3]:
    print(f"  {row['zone_id']} -> {row.get('site_id', 'none')}  dist={row.get('distance_right', 'n/a')}")

# 5. Dissolve by risk tier
dissolved = zones_frame.dissolve(by="risk_tier", aggregations={"composite_risk": "mean", "pop_density": "sum"})
print("\nDissolved by risk tier:")
for row in dissolved.to_records():
    print(f"  {row['risk_tier']}: zones dissolved")

# 6. Raster algebra: combine heat + drought rasters
RASTER_TRANSFORM = (-98.60, 39.80, 0.10, 0.10)
heat_raster = {
    "data": [[0.7, 0.8, 0.9, 0.8, 0.7], [0.6, 0.75, 0.85, 0.78, 0.65],
             [0.5, 0.65, 0.78, 0.72, 0.60], [0.45, 0.60, 0.70, 0.65, 0.55],
             [0.40, 0.55, 0.65, 0.60, 0.50]],
    "transform": RASTER_TRANSFORM,
}
drought_raster = {
    "data": [[0.5, 0.6, 0.7, 0.65, 0.55], [0.55, 0.7, 0.8, 0.75, 0.60],
             [0.60, 0.75, 0.85, 0.78, 0.65], [0.55, 0.70, 0.78, 0.72, 0.60],
             [0.50, 0.62, 0.70, 0.65, 0.55]],
    "transform": RASTER_TRANSFORM,
}
combined_risk_raster = raster_algebra(heat_raster, drought_raster, operation="add")
print(f"\nCombined climate risk raster center: {combined_risk_raster['data'][2][2]:.3f}")

# 7. Query bounds: filter high-risk northern zone
north_zones = zones_frame.query_bounds(-98.50, 39.30, -97.40, 39.70)
print(f"Zones in northern band: {[r['zone_id'] for r in north_zones.to_records()]}")

write_geojson(OUTPUT_DIR / "d5-gp-zones.geojson", zones_frame)
print("\nFrame summary:")
print(json.dumps(zones_frame.summary(), indent=2, default=str))

# Inline visualization
records = zones_frame.to_records()
lons = [float(r["geometry"]["coordinates"][0]) for r in records]
lats = [float(r["geometry"]["coordinates"][1]) for r in records]
risks = [float(r["composite_risk"]) for r in records]
labels = [str(r["zone_id"]) for r in records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc = axes[0].scatter(lons, lats, c=risks, cmap="YlOrRd", s=200, edgecolors="#333", zorder=5)
for lon, lat, lbl, risk in zip(lons, lats, labels, risks):
    axes[0].annotate(f"{lbl}\n({risk:.2f})", (lon, lat), textcoords="offset points", xytext=(5, 5), fontsize=9)
slons = [float(r["geometry"]["coordinates"][0]) for r in ADAPTATION_SITES]
slats = [float(r["geometry"]["coordinates"][1]) for r in ADAPTATION_SITES]
axes[0].scatter(slons, slats, c="blue", s=220, marker="*", zorder=6, label="Adaptation Site")
plt.colorbar(sc, ax=axes[0], label="Composite Climate Risk")
axes[0].set_title("Climate Risk Map (star=adaptation site)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(combined_risk_raster["data"], cmap="RdYlGn_r", origin="upper")
plt.colorbar(im, ax=axes[1], label="Combined Risk (heat+drought)")
axes[1].set_title("Combined Climate Risk Raster")
plt.suptitle("D5 Climate: GeoPromptFrame + Raster Analysis", fontweight="bold")
plt.tight_layout(); plt.show()


## Section C: Scenario Comparison

In [ ]:
baseline   = {"annual_loss_musd": 58.0, "resilience_index": 0.44, "service_reliability": 0.82}
adaptation = {"annual_loss_musd": 32.0, "resilience_index": 0.68, "service_reliability": 0.91}
report = build_scenario_report(baseline, adaptation, higher_is_better=["resilience_index", "service_reliability"])
report_path = export_scenario_report(report, OUTPUT_DIR / "d5-gp-scenario-report.json")
print("Scenario report:", report_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics = list(baseline.keys())
x = range(len(metrics))
width = 0.38
axes[0].bar([i - width/2 for i in x], [float(baseline[m]) for m in metrics], width=width, label="Baseline", color="#94a3b8")
axes[0].bar([i + width/2 for i in x], [float(adaptation[m]) for m in metrics], width=width, label="Adaptation", color="#2563eb")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(metrics, rotation=15)
axes[0].set_title("Baseline vs Adaptation"); axes[0].legend(); axes[0].grid(True, axis="y", alpha=0.3)

delta = [round((adaptation[m] - baseline[m]) / abs(baseline[m]) * 100, 1) for m in metrics]
bar_colors = ["#27ae60" if d > 0 else "#c0392b" for d in delta]
axes[1].barh(metrics, delta, color=bar_colors)
axes[1].axvline(0, color="#555", linewidth=1)
axes[1].set_xlabel("% Change vs Baseline"); axes[1].set_title("Improvement (positive=better)")
axes[1].grid(True, axis="x", alpha=0.3)
plt.suptitle("D5 Climate: Adaptation Scenario Analysis", fontweight="bold")
plt.tight_layout(); plt.show()

(OUTPUT_DIR / "d5-gp-complex.json").write_text(
    json.dumps({"baseline": baseline, "adaptation": adaptation}, indent=2, default=str), encoding="utf-8"
)
print("Wrote d5-gp-complex.json")
